# Build subway scheduled-service supply from GTFS snapshots [WP-A1, Path 2] — rev.2

Turns **dated static GTFS snapshots** into a subway *scheduled train-trips* supply series using
the validated `gtfs_supply.py` engine (keep it beside this notebook).

**rev.2 fixes** (after the current-feed test returned an empty monthly series):
- monthly step-fill now compares by **month**, not exact date (a mid-month snapshot no longer
  drops its own month);
- the monthly `value` is now a **day-type-composed monthly total** (weekday/Sat/Sun levels applied
  to each month's actual calendar), matching how SIR and LIRR supply were built;
- a clear warning when there is only one snapshot (one anchor level ≠ a trajectory).

**You need dated/historical snapshots** for a trajectory. The current feed alone gives one anchor
(2026). The prize is a **2019** vintage (subway is the anchor mode with observed 2019 demand →
structural-gap comparison). Minimum viable: 2019 + current; better: 2019/2022/current.

**Lead worth checking first (may beat the physical-office hunt):** data.ny.gov now hosts
`g8es-h7gb` "MTA Subway Schedules: 2026" (stop-level base+supplement schedule). If prior-year
siblings exist ("...: 2025", "...: 2023"), that is a portal-native historical source for scheduled
trips — query the Socrata catalog for "MTA Subway Schedules". If not, use dated GTFS zips here.


## 0 · Setup — place snapshots, import the tested engine

In [ ]:
import pathlib, datetime as dt, re
import pandas as pd
import gtfs_supply as g          # the unit-tested counting engine

SNAP = pathlib.Path("snapshots"); SNAP.mkdir(exist_ok=True)
RAW  = pathlib.Path("data/raw")
if not RAW.exists(): RAW = pathlib.Path("..")/"data"/"raw"
RAW.mkdir(parents=True, exist_ok=True)

CURRENT_URL = "http://web.mta.info/developers/data/nyct/subway/google_transit.zip"
try:
    import requests
    z = SNAP / f"subway_{dt.date.today():%Y-%m}.zip"
    if not z.exists():
        r = requests.get(CURRENT_URL, timeout=120); r.raise_for_status()
        z.write_bytes(r.content); print("downloaded current feed ->", z)
except Exception as e:
    print("skip current-feed download (offline or moved):", str(e)[:120])

zips = sorted(SNAP.glob("*.zip"))
print("snapshots present:", [p.name for p in zips] or "NONE — drop dated .zip files into snapshots/")


## 1 · Count scheduled subway trips per snapshot
`route_types=(1,)` keeps subway only. **Check for SIR:** if the feed contains a route `SI`/`SIR`
(route_type 1), exclude it so it does not double-count with the SIR supply series. The effective
date is parsed from the filename (`subway_YYYY-MM.zip`).

In [ ]:
def eff_date(path):
    m = re.search(r"(\d{4})-(\d{2})", path.name)
    if not m: raise ValueError(f"name it subway_YYYY-MM.zip; got {path.name}")
    return dt.date(int(m.group(1)), int(m.group(2)), 15)

rows = []
for z in zips:
    d = eff_date(z)
    rc = g.representative_counts(str(z), d, route_types=(1,))
    rows.append({"effective": d, "weekday": rc["weekday"], "saturday": rc["saturday"], "sunday": rc["sunday"]})
    print(f"{z.name}: weekday={rc['weekday']:>5}  sat={rc['saturday']:>5}  sun={rc['sunday']:>5}")
snaps = pd.DataFrame(rows).sort_values("effective").reset_index(drop=True)
snaps


## 2 · Sanity check
Subway weekday scheduled trips are ~8,000 (validated on the 2026 feed: 8,348 weekday / 6,090 Sat /
5,530 Sun; weekend<weekday as expected). Hundreds, or erratic swings, signal a Supplemented feed or
a parse error. Note any published NYCT figure you have rather than asserting one.

In [ ]:
if len(snaps) >= 2:
    chg = snaps["weekday"].iloc[-1]/snaps["weekday"].iloc[0] - 1
    print(f"weekday scheduled trips first->last snapshot: {snaps['weekday'].iloc[0]} -> "
          f"{snaps['weekday'].iloc[-1]} ({chg:+.1%})")
print("Levels should be ~thousands and change only modestly across the recovery (no GCM-scale move).")


## 3 · Monthly step-series (day-type-composed totals), target shape
Each snapshot's day-type levels hold until the next snapshot. For each month we compose a
**monthly total** = Σ over the month's days of the applicable weekday/Sat/Sun level — the same
monthly-total construction as SIR and LIRR supply. Months before the earliest snapshot are left
empty on purpose (no known schedule). Emits `derived_subway_supply` rows.

In [ ]:
START, END = "2016-01-01", f"{dt.date.today():%Y-%m-01}"
months = pd.date_range(START, END, freq="MS")
snaps["eff_month"] = pd.to_datetime(snaps["effective"]).values.astype("datetime64[M]")

def compose_total(ms, row):
    days = pd.date_range(ms, ms + pd.offsets.MonthEnd(0), freq="D")
    lvl = {0:"weekday",1:"weekday",2:"weekday",3:"weekday",4:"weekday",5:"saturday",6:"sunday"}
    return int(sum(row[lvl[d.weekday()]] for d in days))   # holidays: minor, refine later

vals = {}
for m in months:
    prior = snaps[snaps["eff_month"] <= m]
    if len(prior):
        vals[m] = compose_total(m, prior.iloc[-1])

out = (pd.Series(vals, name="value").rename_axis("period_start").reset_index())
if out.empty:
    print("EMPTY: no month is on/after any snapshot. With one CURRENT snapshot you get ~1 month.\n"
          "This is expected — add DATED historical snapshots (2019/2022) for a real trajectory.")
else:
    out = out.assign(grain="monthly", service="Subway Scheduled Service", mode_class="subway",
                     measure_type="scheduled_service", unit="train_trips",
                     data_basis="derived_subway_supply", evidence="derived",
                     period_label=out["period_start"].dt.strftime("%Y-%m"))
    out = out[["grain","service","mode_class","measure_type","unit","data_basis",
               "period_start","period_label","value","evidence"]]
    out.to_csv(RAW/"subway_scheduled_service_monthly_preview.csv", index=False)
    print(f"rows: {len(out)} | span {out.period_start.min().date()}..{out.period_start.max().date()}")
    if len(snaps) == 1:
        print("WARNING: only ONE snapshot -> a single anchor level, not a trajectory. "
              "Add a 2019 (and ideally 2022) vintage to snapshots/ and re-run.")
    print(out.head(3).to_string(index=False)); print("..."); print(out.tail(3).to_string(index=False))


## 4 · Next steps
Once you have ≥2 snapshots (ideally a 2019 vintage):
1. Append the preview to `data/feeds/mta_chart_feed_multigrain.csv`; compute variation-stat columns.
2. `src/config.py`: `SUBWAY_SUPPLY = "Subway Scheduled Service"`; add `derived_subway_supply` to the
   tier list in `docs/methodology_provenance.md` + data dictionary.
3. Extend the supply-vs-demand figure: a subway panel indexed to **2019** (the structural-gap prize —
   subway demand ~0.76 of 2019; is scheduled service held near 2019, or cut toward demand?).
4. Label: subway = *scheduled* (own axis), distinct from LIRR *operated*. Keep subway-only
   (`route_type=1`), exclude SIR. Record snapshot dates in `DECISION_REGISTER.md`.

**Honest expectation:** subway scheduled service changed far less than LIRR (no GCM-scale move), so
the level effect is likely modest — the value is the 2019 anchor comparison and completing the index
across three modes (SIR, LIRR, subway).
